# Pointer-generator results

This lightweight notebook reads the downloaded generator ZIP. It performs no training,
checkpoint loading, inference, or test-set access.

## Step 1 — Load the artifact


In [32]:
import json,csv,io,zipfile,sys
from pathlib import Path
from IPython.display import display,Markdown
NAME="pointer_generator_artifacts.zip"
candidates=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_pointer_generator")/NAME]
archive=next((p for p in candidates if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive"); archive=next((p for p in candidates if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); names=set(zf.namelist())
required={"generator_results/runs.csv","generator_results/generator_summary.json"}
assert required<=names,f"Missing {required-names}"
summary=json.loads(zf.read("generator_results/generator_summary.json"))
print("Loaded",archive,"— no training, model loading, or inference")


Loaded /content/pointer_generator_artifacts.zip — no training, model loading, or inference


## Step 2 — Compare generation and extraction


In [33]:
s=summary["slices"]
lines=["| Validation slice | n | BLEU-1 | METEOR | ROUGE-L | EM | F1 |","|---|---:|---:|---:|---:|---:|---:|"]
for k,v in s.items(): lines.append(f"| {k} | {v['n']} | {v['bleu1']:.2f} | {v['meteor']:.2f} | {v['rougeL']:.2f} | {v['em']:.2f} | {v['f1']:.2f} |")
display(Markdown("## Validation results — one seed\n\n"+"\n".join(lines)))
overall=s["all"]
display(Markdown(f"""The from-scratch pointer-generator obtains **ROUGE-L {overall['rougeL']:.2f}**.
For context, the previous three-seed extractive v2 experiment obtained 44.22 ± 0.16, while the
pretrained T5 baseline obtained 56.93. Those are historical references rather than paired runs."""))


## Validation results — one seed

| Validation slice | n | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|---:|
| all | 1076 | 31.24 | 25.94 | 35.57 | 21.19 | 35.77 |
| fully_extractive | 659 | 45.31 | 35.91 | 49.70 | 34.45 | 49.69 |
| partially_or_non_extractive | 417 | 9.00 | 10.19 | 13.24 | 0.24 | 13.77 |
| no_good_span | 191 | 1.90 | 4.33 | 2.88 | 0.52 | 3.14 |

The from-scratch pointer-generator obtains **ROUGE-L 35.57**.
For context, the previous three-seed extractive v2 experiment obtained 44.22 ± 0.16, while the
pretrained T5 baseline obtained 56.93. Those are historical references rather than paired runs.

## Qualitative non-extractive cases

## Step 3 — Inspect extractive-status slices


In [34]:
xs=summary["examples"]
nonext=[x for x in xs if x["best_span_f1"]<100]
best=sorted(nonext,key=lambda x:x["f1"],reverse=True)[:12]
worst=sorted(nonext,key=lambda x:x["f1"])[:12]
def show(items,title):
    lines=["| F1 | Best extractive F1 | Question | Gold | Prediction | Tokens absent from tweet |","|---:|---:|---|---|---|---|"]
    for x in items:
        clean=lambda y:str(y).replace("|","/").replace("\n"," ")
        lines.append(f"| {x['f1']:.1f} | {x['best_span_f1']:.1f} | {clean(x['question'])} | {clean(x['gold'])} | {clean(x['prediction'])} | {clean(x['generated_tokens'])} |")
    display(Markdown("### "+title+"\n\n"+"\n".join(lines)))
show(best,"Strongest partially/non-extractive cases")
show(worst,"Weakest partially/non-extractive cases")


### Strongest partially/non-extractive cases

| F1 | Best extractive F1 | Question | Gold | Prediction | Tokens absent from tweet |
|---:|---:|---|---|---|---|
| 100.0 | 0.0 | tweet is not visible | ['tweet is not visible'] | tweet is not visible | ['tweet', 'is', 'not', 'visible'] |
| 88.9 | 88.9 | how long was david lynch in negotiations? | ['1 year 4 months'] | 1 year and 4 months | [] |
| 80.0 | 80.0 | who's car was it that melted? | ['van dyke.'] | van dyke's | [] |
| 80.0 | 80.0 | whose goal was it to give him the hat trick? | ['rick nash'] | rick nash's | [] |
| 80.0 | 80.0 | what time did nashville air? | ['10-9 central'] | 10/9 | [] |
| 80.0 | 80.0 | who make productive teams? | ['men and women.'] | men & women | [] |
| 80.0 | 80.0 | she captured a video of whose show? | ['tom petty'] | tom petty’s | [] |
| 80.0 | 80.0 | who sent an old letter to lebron? | ['dan gilbert'] | dan gilbert's | [] |
| 80.0 | 92.3 | who raved about davis webb? | ['coaches and texas tech and cal'] | texas tech and cal | [] |
| 75.0 | 75.0 | who is he asking that people pray for? | ['racquel and his children.'] | racquel and their children | [] |
| 66.7 | 66.7 | how long has she been in remission from leukemia? | ['12 years'] | 12 | [] |
| 66.7 | 66.7 | how many miles south does the water go? | ['152 miles'] | 152 | [] |

### Weakest partially/non-extractive cases

| F1 | Best extractive F1 | Question | Gold | Prediction | Tokens absent from tweet |
|---:|---:|---|---|---|---|
| 0.0 | 0.0 | who is lin manuel? | ['a twitter user'] | hillary clinton | ['hillary', 'clinton'] |
| 0.0 | 0.0 | what condition does she mention? | ['fast metabolism.'] | <unk> | ['<', 'unk', '>'] |
| 0.0 | 0.0 | how many points were scored in the game? | ['three'] | 3 3 | [] |
| 0.0 | 0.0 | what sport did thomas play? | ['football club.'] | villa | [] |
| 0.0 | 0.0 | what are resources going to be spent on? | ['defeating hilary'] | ultimately defeat hillary | [] |
| 0.0 | 0.0 | what happened during this natural disaster? | ['hurricane hermine'] | <unk> <unk> | ['<', 'unk', '>', '<', 'unk', '>'] |
| 0.0 | 0.0 | why is the president mad? | ['we have no idea'] | for the internet of the internet . | ['for'] |
| 0.0 | 0.0 | who is tim kaine? | ["hillary clinton's running mate"] | <unk> | ['<', 'unk', '>'] |
| 0.0 | 0.0 | what was removed from the site? | ['something inappropriate'] | unacceptable | [] |
| 0.0 | 0.0 | who does the internet want to burn at the stake? | ['rachel dolezal'] | austin von | [] |
| 0.0 | 0.0 | where is this picture from? | ['time 100'] | <unk> | ['<', 'unk', '>'] |
| 0.0 | 0.0 | who opened and closed the debate? | ['jeb bush'] | donald trump | [] |

## Step 4 — Write the conclusion


In [35]:
text=f"""# Pointer-generator experiment

The one-seed from-scratch pointer-generator achieved validation ROUGE-L {overall['rougeL']:.2f},
F1 {overall['f1']:.2f}, and EM {overall['em']:.2f}. Its performance on the
partially/non-extractive and no-good-span slices indicates whether generation and copying
address the extractive model's structural limitation. Because this is one seed and uses a
different output architecture, it is an exploratory result. The locked test set was not used.
"""
Path("pointer_generator_report.md").write_text(text,encoding="utf-8")
display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Pointer-generator experiment

The one-seed from-scratch pointer-generator achieved validation ROUGE-L 35.57,
F1 35.77, and EM 21.19. Its performance on the
partially/non-extractive and no-good-span slices indicates whether generation and copying
address the extractive model's structural limitation. Because this is one seed and uses a
different output architecture, it is an exploratory result. The locked test set was not used.
